# Demand Forecasting — walkthrough

A guided tour of the pipeline in `src/demand_forecasting/`. Every heavy step is a
thin call into the package, so this notebook stays readable and the logic stays
tested. Run top to bottom; the data downloads itself on first use.

In [ ]:
import sys
sys.path.insert(0, "src")
import numpy as np, pandas as pd
from demand_forecasting import data, baselines, causal, plots
from demand_forecasting.models import ETSModel, SarimaModel, LightGBMModel
from demand_forecasting.backtest import backtest, summarize, rolling_origin_splits
np.random.seed(7)

## 1. Data

UCI Online Retail II, aggregated to a continuous daily total-units series for the UK.

In [ ]:
df = data.get_daily_demand()
series = df["units"]
series.describe()

## 2. EDA

Weekly seasonality (Saturday is a structural zero — the store is closed), a
Christmas ramp, and a heavy right tail from large wholesale orders.

In [ ]:
plots.plot_eda(series)
series.groupby(series.index.dayofweek).mean().round(0)

## 3. Baseline

Seasonal-naive (m=7) is the benchmark every model must beat.

In [ ]:
h = 28
snaive = baselines.seasonal_naive(series.iloc[:-h], h)
snaive[:7]

## 4. Backtest

Rolling-origin CV: 6 consecutive 28-day windows, each model trained only on its
past. The closed-Saturday business rule is applied to every forecast.

In [ ]:
models = {
    "ETS": ETSModel().forecast,
    "SARIMA": SarimaModel().forecast,
    "LightGBM": LightGBMModel().forecast,
}
results = backtest(series, models)
summary = summarize(results)
summary.round(3)

In [ ]:
plots.plot_backtest(summary)

**SARIMA wins on every metric.** The LightGBM does not beat the classical
seasonal models on this single rigid-weekly series — an honest finding. ML earns
its keep across many series (global models), not on one smooth aggregate.

## 5. Forecast + intervals + error analysis

In [ ]:
fold = list(rolling_origin_splits(series))[-1]
fc = SarimaModel().forecast_interval(fold.train, len(fold.test))
from demand_forecasting.postprocess import apply_closed_days
mean = apply_closed_days(fc.mean, fold.test.index)
plots.plot_forecast(fold.train, fold.test, mean,
                    apply_closed_days(fc.lower, fold.test.index),
                    apply_closed_days(fc.upper, fold.test.index))
plots.plot_residuals(fold.test, mean)

## 6. Causal bonus — recover a known promo uplift via DiD

Simulate a +15% promo on treated stores in a synthetic panel and recover it with
a two-way fixed-effects difference-in-differences.

In [ ]:
did = causal.run_promo_study(series, uplift=0.15)
print(f"true 15%  est {did.estimated_uplift:.1%}  "
      f"CI [{did.ci_low:.1%}, {did.ci_high:.1%}]  recovered={did.recovered}")